# Attention from Scratch


 **Scaled Dot-Product Attention** 
 $$\boxed{\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V}$$

 ## Dimensional Analysis of $\text{Attention}(Q, K, V)$

$$Q \in \mathbb{R}^{T \times d_k} \quad K \in \mathbb{R}^{T \times d_k} \quad V \in \mathbb{R}^{T \times d_v}$$

$T$ : sequence length (number of tokens in the sequence)

**Softmax** : Applied along the last dimension (over the $T$ keys for each query)  

**Result: $\mathbb{R}^{T \times d_v}$** : same shape as $V$. Each token is replaced by a weighted sum of the values.

**Remark : in practice with batching**

In practice, inputs have an extra batch dimension $B$ (number of examples processed in parallel):

$$Q \in \mathbb{R}^{B \times T \times d_k} \quad K \in \mathbb{R}^{B \times T \times d_k} \quad V \in \mathbb{R}^{B \times T \times d_v}$$

The computation is identical : PyTorch applies the matrix multiplications independently for each of the $B$ examples. The final output is $\mathbb{R}^{B \times T \times d_v}$. The $B$ dimension does not change the mathematics, only the parallelism.

## Why divide by $\sqrt{d_k}$?

**Assumptions**

The components of $Q$ and $K$ are i.i.d. $\sim \mathcal{N}(0, 1)$.

**Objective**

We want $\forall i, j$:

$$\frac{(QK^\top)_{ij}}{\alpha} \sim \mathcal{N}(0, 1)$$

which is equivalent to finding $\alpha$ such that:

$$\text{Var}\left(\frac{(QK^\top)_{ij}}{\alpha}\right) = 1$$

**Derivation**

By the variance scaling property:

$$\frac{\text{Var}\left((QK^\top)_{ij}\right)}{\alpha^2} = 1 \implies \alpha^2 = \text{Var}\left((QK^\top)_{ij}\right)$$

By the Cauchy matrix multiplication formula:

$$(QK^\top)_{ij} = \sum_{l=1}^{d_k} Q_{il} K_{jl}$$

The terms $Q_{il} K_{jl}$ are independent. For each one:

$$\mathbb{E}[Q_{il} K_{jl}] = \mathbb{E}[Q_{il}]\mathbb{E}[K_{jl}] = 0$$

$$\text{Var}(Q_{il} K_{jl}) = \mathbb{E}[Q_{il}^2]\mathbb{E}[K_{jl}^2] = 1 \times 1 = 1$$

By additivity of variances of independent variables:

$$\text{Var}\left((QK^\top)_{ij}\right) = \sum_{l=1}^{d_k} \text{Var}(Q_{il} K_{jl}) = d_k$$

**Conclusion**

$$\alpha^2 = d_k \implies \alpha = \sqrt{d_k}$$

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device = {device}")

## 2. Scaled Dot-Product Attention


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q(B,T,dk) K(B,T,dk) V(B,T,dv) mask(B,T,T)
    
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output, attn_weights

In [ ]:
B = 128
T = 10 
d_k = 16
d_v = 16

Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(output.shape)        
print(attn_weights.shape) 
print(attn_weights[0].sum(dim=-1))  

In [ ]:
plt.imshow(attn_weights[0].detach(), cmap='viridis', origin="lower")
plt.colorbar()
plt.title('Attention weights (batch 0)')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.show()

In [ ]:
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # upper triangle = True
mask = mask.unsqueeze(0).expand(B, -1, -1)   # (B, T, T)
output_masked, attn_masked = scaled_dot_product_attention(Q, K, V, mask)

plt.imshow(attn_masked[0].detach(), cmap='viridis', origin="lower")
plt.colorbar()
plt.title('Attention weights with mask (batch 0)')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.show()

## 3. Positional Encoding

**Why it is necessary:** attention is permutation-equivariant : if you shuffle the input tokens, the output is shuffled the same way. Nothing in the attention formula encodes the position of a token in the sequence. Positional encoding fixes this by adding a position-dependent signal to the embeddings before the attention layers.

$$x_\text{input} = \text{Embedding}(token) + PE_{pos}$$


**The sinusoidal formula** :

$$PE_{(pos,\, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

$$PE_{(pos,\, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

where $pos$ is the position in the sequence and $i$ is the dimension index.

or example, with $d_\text{model} = 8$ and position $pos$:

$$PE_{pos} = \begin{bmatrix} \sin(\omega_0 \cdot pos) \\ \cos(\omega_0 \cdot pos) \\ \sin(\omega_1 \cdot pos) \\ \cos(\omega_1 \cdot pos) \\ \sin(\omega_2 \cdot pos) \\ \cos(\omega_2 \cdot pos) \\ \sin(\omega_3 \cdot pos) \\ \cos(\omega_3 \cdot pos) \end{bmatrix}$$


### Why Sin and Cos — Linear Representation of Relative Positions

The key property: $PE_{pos+k}$ is a linear function of $PE_{pos}$.

For a pair of dimensions $(2i, 2i+1)$ at frequency $\omega$, using the angle addition formulas:

$$\sin(\omega(pos+k)) = \sin(\omega \cdot pos)\cos(\omega k) + \cos(\omega \cdot pos)\sin(\omega k)$$

$$\cos(\omega(pos+k)) = \cos(\omega \cdot pos)\cos(\omega k) - \sin(\omega \cdot pos)\sin(\omega k)$$

In matrix form:

$$PE_{pos+k} = \underbrace{\begin{bmatrix} \cos(\omega k) & \sin(\omega k) \\ -\sin(\omega k) & \cos(\omega k) \end{bmatrix}}_{M_k} \cdot PE_{pos}$$

$M_k$ is a rotation matrix that depends only on the offset $k$, not on the absolute position $pos$. Moving from position $pos$ to position $pos+k$ is a rotation of angle $\omega k$ in the $(\sin, \cos)$ plane — the same rotation regardless of where you are in the sequence.

This is why the model can easily learn relative positions: "token at distance $k$" = apply $M_k$ to the positional encoding, which is a simple matrix multiplication.

In [ ]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model, max_len=5000, dropout=0.1): 
        
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pos = torch.arange(max_len).unsqueeze(1)   #pos(max_len,1)
    
        i = torch.arange(0, d_model, 2)     #i(d_model/2,)
        div_term = torch.exp(i * (-math.log(10000.0) / d_model)) #div_term(d_model/2,)
    
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div_term)   
        pe[:, 1::2] = torch.cos(pos * div_term)    
        pe = pe.unsqueeze(0)                        
        self.register_buffer('pe', pe )

    def forward(self, x):  #x(B,T,d_model)
        T = x.size(1)
        x = x + self.pe[:, :T]   
        return self.dropout(x)
      

In [ ]:
d_model = 256 
T = 10
B = 128 
pe = PositionalEncoding(d_model, dropout=0.0) #pe(1,max_len,d_model)
x = torch.zeros(B, T, d_model)
out = pe(x)
print(out.shape)   
# out[0] is now the PE matrix itself (since x=0)

## 4. Multi-Head Attention

Instead of a single attention with $d_\text{model}$-dimensional Q/K/V, project $h$ times into subspaces of dimension $d_k = d_\text{model}/h$, run attention in parallel on each projection, then concatenate and project back:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$

$$\text{head}_i = \text{Attention}(xW^Q_i,\ xW^K_i,\ xW^V_i)$$

Projection matrices:

$$W^Q_i \in \mathbb{R}^{d_\text{model} \times d_k} \quad W^K_i \in \mathbb{R}^{d_\text{model} \times d_k} \quad W^V_i \in \mathbb{R}^{d_\text{model} \times d_v} \quad W^O \in \mathbb{R}^{d_\text{model} \times d_\text{model}}$$

In the paper: $h = 8$, $d_\text{model} = 512$, $d_k = d_v = 64$.

In practice, the $h$ projections are batched into a single matrix of shape $(d_\text{model}, h \cdot d_k) = (d_\text{model}, d_\text{model})$ — one `nn.Linear(d_model, d_model)` for each of $W^Q$, $W^K$, $W^V$, $W^O$ — then reshape to separate the heads.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # TODO: check that d_model is divisible by num_heads
        # TODO: set self.d_k = d_model // num_heads
        # TODO: store self.num_heads and self.d_model
        # TODO: define the four linear projections (no bias needed):
        #   self.W_Q = nn.Linear(d_model, d_model, bias=False)
        #   self.W_K = nn.Linear(d_model, d_model, bias=False)
        #   self.W_V = nn.Linear(d_model, d_model, bias=False)
        #   self.W_O = nn.Linear(d_model, d_model, bias=False)
        pass

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q, K, V: (B, T, d_model)
            mask: (B, T, T) or None
        Returns:
            output: (B, T, d_model)
            attn_weights: (B, num_heads, T, T)
        """
        # TODO Step 1: get batch size B and sequence length T from Q.shape

        # TODO Step 2: project Q, K, V through W_Q, W_K, W_V
        #   q = self.W_Q(Q)  ->  (B, T, d_model)
        #   k = self.W_K(K)
        #   v = self.W_V(V)

        # TODO Step 3: split into heads
        #   reshape (B, T, d_model) -> (B, T, num_heads, d_k) -> (B, num_heads, T, d_k)
        #   q = q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        #   same for k, v

        # TODO Step 4: prepare mask for multiple heads
        #   if mask is not None: mask = mask.unsqueeze(1)  # (B, 1, T, T) broadcasts over heads

        # TODO Step 5: run scaled_dot_product_attention
        #   q, k, v have shape (B, num_heads, T, d_k) — the function works since it uses transpose(-2, -1)
        #   attn_output, attn_weights = scaled_dot_product_attention(q, k, v, mask)
        #   attn_output shape: (B, num_heads, T, d_k)

        # TODO Step 6: concatenate heads back
        #   transpose to (B, T, num_heads, d_k) then view as (B, T, d_model)
        #   attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # TODO Step 7: apply output projection W_O
        #   output = self.W_O(attn_output)

        # TODO: return output, attn_weights
        pass

In [ ]:
# TODO: test MultiHeadAttention
# B, T, d_model, num_heads = 2, 5, 64, 4  -> d_k = 16
# x = torch.randn(B, T, d_model)
# mha = MultiHeadAttention(d_model, num_heads)
# out, attn = mha(x, x, x)
# print(out.shape)   # should be (2, 5, 64)
# print(attn.shape)  # should be (2, 4, 5, 5)

In [ ]:
# TODO: verify against nn.MultiheadAttention from PyTorch
# Use batch_first=True so input shape matches (B, T, d_model)
# Copy the weights from your MultiHeadAttention into nn.MultiheadAttention
# Check that outputs are identical (or very close, up to floating point)

## 5. TransformerBlock

One encoder block from the paper (Figure 1, left):

$$x' = \text{LayerNorm}(x + \text{MultiHead}(x, x, x))$$
$$y = \text{LayerNorm}(x' + \text{FFN}(x'))$$

The Feed-Forward Network is two linear layers with ReLU:

$$\text{FFN}(x) = \max(0,\ xW_1 + b_1)W_2 + b_2$$

with inner dimension $d_\text{ff} = 4 \times d_\text{model}$ in the original paper.

**Why $d_\text{ff} > d_\text{model}$?**  In the larger $d_\text{ff}$-dimensional space, the ReLU can selectively activate or deactivate many more directions. Attention aggregates information across tokens; the FFN processes each token individually in this enriched space. In modern LLMs, the FFN is interpreted as an associative memory — the weights $W_1, W_2$ store key-value associations learned during training.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO: self.attn = MultiHeadAttention(d_model, num_heads)
        # TODO: self.ff = nn.Sequential(
        #           nn.Linear(d_model, d_ff),
        #           nn.ReLU(),
        #           nn.Linear(d_ff, d_model)
        #       )
        # TODO: self.norm1 = nn.LayerNorm(d_model)
        # TODO: self.norm2 = nn.LayerNorm(d_model)
        # TODO: self.dropout = nn.Dropout(dropout)
        pass

    def forward(self, x, mask=None):
        """
        Args:
            x: (B, T, d_model)
            mask: (B, T, T) or None
        Returns:
            (B, T, d_model)
        """
        # TODO: self-attention sub-layer (Q=K=V=x)
        #   attn_out, _ = self.attn(x, x, x, mask)
        #   x = self.norm1(x + self.dropout(attn_out))

        # TODO: feed-forward sub-layer
        #   ff_out = self.ff(x)
        #   x = self.norm2(x + self.dropout(ff_out))

        # TODO: return x
        pass

In [ ]:
# TODO: test TransformerBlock
# B, T, d_model, num_heads, d_ff = 2, 5, 64, 4, 256
# x = torch.randn(B, T, d_model)
# block = TransformerBlock(d_model, num_heads, d_ff)
# y = block(x)
# print(y.shape)   # should be (2, 5, 64) — same as input

## 6. TransformerEncoder

Stack $N$ TransformerBlocks. Each block takes the output of the previous one as input. In the original paper $N = 6$.

Use `nn.ModuleList` (not a plain Python list) so PyTorch tracks the parameters of all blocks.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        # TODO: self.layers = nn.ModuleList([
        #           TransformerBlock(d_model, num_heads, d_ff, dropout)
        #           for _ in range(num_layers)
        #       ])
        pass

    def forward(self, x, mask=None):
        # TODO: pass x through each block in self.layers
        #   for layer in self.layers:
        #       x = layer(x, mask)
        #   return x
        pass

In [ ]:
# TODO: test TransformerEncoder
# B, T, d_model, num_heads, d_ff, num_layers = 2, 5, 64, 4, 256, 4
# x = torch.randn(B, T, d_model)
# encoder = TransformerEncoder(d_model, num_heads, d_ff, num_layers)
# y = encoder(x)
# print(y.shape)   # should be (2, 5, 64)
# print("n_params:", sum(p.numel() for p in encoder.parameters()))

## 7. Empirical Verification: Softmax Saturation

Verify empirically that without the $\sqrt{d_k}$ scaling, the variance of $QK^\top$ grows linearly with $d_k$ and the softmax saturates.

In [ ]:
# TODO: verify the variance claim empirically
# for d_k in [4, 16, 64, 256, 1024]:
#     q = torch.randn(10000, d_k)
#     k = torch.randn(10000, d_k)
#     dots = (q * k).sum(dim=-1)
#     print(f"d_k={d_k:4d}  Var(q.k)={dots.var().item():7.2f}  Var(q.k/sqrt(d_k))={ (dots / math.sqrt(d_k)).var().item():.3f}")

In [ ]:
# TODO: visualize softmax saturation
# Plot softmax([s, 0, 0, ..., 0]) as s increases from 0 to 10
# Show that the output becomes a one-hot vector

## 8. Shape Verification Summary

| Module | Input shape | Output shape |
|--------|------------|-------------|
| `scaled_dot_product_attention` | Q,K,V: $(B, T, d_k)$ | $(B, T, d_v)$, $(B, T, T)$ |
| `PositionalEncoding` | $(B, T, d_\text{model})$ | $(B, T, d_\text{model})$ |
| `MultiHeadAttention` | $(B, T, d_\text{model})$ | $(B, T, d_\text{model})$ |
| `TransformerBlock` | $(B, T, d_\text{model})$ | $(B, T, d_\text{model})$ |
| `TransformerEncoder` | $(B, T, d_\text{model})$ | $(B, T, d_\text{model})$ |